In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import plotly.graph_objects as go
import logging
import ast
import math
from collections import Counter

from torch.nn.utils.rnn import pad_sequence

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import utils
import z_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(device)

cuda


# 1. Data

In [4]:
spike_df = pd.read_csv('../../Data_processed/spike_tensors_x.csv')

spike_df['spike_durations'] = spike_df['spike_durations'].apply(ast.literal_eval)
spike_df['spike_durations'] = spike_df['spike_durations'].apply(lambda x: x + [0]*(4-len(x)))

spike_df['spike_magnitudes'] = spike_df['spike_magnitudes'].apply(ast.literal_eval)
spike_df['spike_magnitudes'] = spike_df['spike_magnitudes'].apply(lambda x: x + [0.0]*(4-len(x)))

spike_df['time'] = spike_df['spike_times_intervals'].apply(ast.literal_eval)
spike_df['time'] = spike_df['time'].apply(lambda x: x + [0]*(4-len(x)))

print(spike_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3     [2, 2, 3]    [3, 3, 4, 0]    [0.663, 0.493, 0.886, 0.0]   
1          2     [3, 2, 3]    [4, 5, 0, 0]      [0.933, 1.306, 0.0, 0.0]   
2          2     [2, 2, 3]    [6, 9, 0, 0]      [1.191, 1.085, 0.0, 0.0]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]    [3, 3, 5, 0]    [0.408, 0.991, 1.413, 0.0]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [5]:
# Get the probability of different spike num
spike_num = spike_df['spike_num'].value_counts()
spike_num = spike_num.sort_index()
total = spike_num.sum()
spike_num = spike_num / total
prob_df = pd.DataFrame(spike_num)
prob_df.columns = ['probability']
prob_df= prob_df.reset_index()
prob_df = prob_df.sort_values(by='probability', ascending=False)

print(prob_df)


    spike_num  probability
2           2     0.305778
3           3     0.269052
1           1     0.157257
4           4     0.133918
0           0     0.083342
5           5     0.040030
6           6     0.008242
7           7     0.001684
8           8     0.000406
9           9     0.000167
10         10     0.000086
11         11     0.000032
12         12     0.000007


In [6]:
# Only keep the days with spike_num < 5
spike_df = spike_df[spike_df['spike_num'] < 5]
spike_num = spike_df['spike_num'].value_counts()
spike_num = spike_num.sort_index()
total = spike_num.sum()
spike_num = spike_num / total
prob_df = pd.DataFrame(spike_num)
prob_df.columns = ['probability']
prob_df= prob_df.reset_index()
prob_df = prob_df.sort_values(by='probability', ascending=False)

print(prob_df)


   spike_num  probability
2          2     0.322093
3          3     0.283407
1          1     0.165647
4          4     0.141064
0          0     0.087789


In [7]:
all_durations = [duration for sublist in spike_df['spike_durations'] for duration in sublist if duration != 0]

# Count the occurrences of each duration
duration_counts = Counter(all_durations)

# Calculate the total number of non-zero durations
total_durations = sum(duration_counts.values())

# Calculate the probability of each duration
duration_probabilities = {duration: count / total_durations for duration, count in duration_counts.items()}

# Convert to a DataFrame for better readability
probability_df = pd.DataFrame(list(duration_probabilities.items()), columns=['Duration', 'Probability'])

# Sort by Duration
probability_df = probability_df.sort_values(by='Probability', ascending=False).reset_index(drop=True)

print(probability_df)

# Add together the probabilities for 10 most common durations
top_10_probability = probability_df['Probability'].iloc[:13].sum()
print(f'The 13 most common durations account for {top_10_probability:.2%} of all non-zero durations')


    Duration   Probability
0          3  4.396302e-01
1          5  1.011166e-01
2          4  8.101210e-02
3          6  7.845331e-02
4          7  5.036503e-02
5          8  4.227612e-02
6          9  3.424870e-02
7         10  2.768852e-02
8         11  2.366418e-02
9         12  1.988363e-02
10        13  1.642147e-02
11        14  1.374648e-02
12        15  1.110663e-02
13         2  8.598698e-03
14        16  8.511165e-03
15         1  8.457382e-03
16        17  6.150865e-03
17        18  4.768835e-03
18        19  3.672365e-03
19        20  2.994449e-03
20        21  2.450606e-03
21        22  2.064415e-03
22        23  1.746647e-03
23        24  1.510093e-03
24        25  1.252272e-03
25        26  1.116042e-03
26        27  9.972252e-04
27        28  9.186307e-04
28        29  8.297111e-04
29        30  7.367846e-04
30        31  6.717516e-04
31        32  5.782087e-04
32        33  4.507624e-04
33        34  3.555244e-04
34        48  3.054397e-04
35        35  2.535057e-04
3

In [8]:
top_durations = set(probability_df.nlargest(7, 'Probability')['Duration'])

# Function to filter and reorder durations in each row
def filter_and_reorder_with_indices(duration_list, magnitude_list, time_list):
    # Ensure the lists are of equal length
    min_length = 4
    
    # Trim the lists if they have inconsistent lengths
    duration_list = duration_list[:min_length]
    magnitude_list = magnitude_list[:min_length]
    time_list = time_list[:min_length]
    
    # Indices of the durations to keep based on the top probabilities
    keep_indices = [i for i, d in enumerate(duration_list) if d in top_durations]

    # Filter and keep only the relevant indices, moving zeros to the end
    filtered_durations = [duration_list[i]-2 for i in keep_indices] + [0] * (min_length - len(keep_indices))
    filtered_magnitudes = [magnitude_list[i] for i in keep_indices] + [0.0] * (min_length - len(keep_indices))
    filtered_time = [time_list[i] for i in keep_indices] + [0] * (min_length - len(keep_indices))

    return filtered_durations, filtered_magnitudes, filtered_time

spike_df[['filtered_spike_durations', 'filtered_spike_magnitudes', 'filtered_time']] = spike_df.apply(
    lambda row: filter_and_reorder_with_indices(row['spike_durations'], row['spike_magnitudes'], row['time']),
    axis=1, result_type='expand'
)

print(spike_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3     [2, 2, 3]    [3, 3, 4, 0]    [0.663, 0.493, 0.886, 0.0]   
1          2     [3, 2, 3]    [4, 5, 0, 0]      [0.933, 1.306, 0.0, 0.0]   
2          2     [2, 2, 3]    [6, 9, 0, 0]      [1.191, 1.085, 0.0, 0.0]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]    [3, 3, 5, 0]    [0.408, 0.991, 1.413, 0.0]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [9]:
all_durations = [duration for sublist in spike_df['filtered_spike_durations'] for duration in sublist if duration != 0]
# Count the occurrences of each duration
duration_counts = Counter(all_durations)

# Calculate the total number of non-zero durations
total_durations = sum(duration_counts.values())

# Calculate the probability of each duration
duration_probabilities = {duration: count / total_durations for duration, count in duration_counts.items()}

# Convert to a DataFrame for better readability
probability_df = pd.DataFrame(list(duration_probabilities.items()), columns=['Duration', 'Probability'])

# Sort by Duration
probability_df = probability_df.sort_values(by='Probability', ascending=False).reset_index(drop=True)

print(probability_df)

   Duration  Probability
0         1     0.531531
1         3     0.122254
2         2     0.097947
3         4     0.094853
4         5     0.060893
5         6     0.051114
6         7     0.041408


In [10]:
def count_non_zero_durations(duration_list):
    return sum(1 for d in duration_list if d != 0)

# Update the spike_num column to reflect the count of non-zero durations
spike_df['spike_num'] = spike_df['filtered_spike_durations'].apply(count_non_zero_durations)

# Display the first few rows to verify the changes
spike_df.head(20)

,spike_num,spike_type,spike_durations,spike_magnitudes,ID-statistics,weather,spike_times_intervals,date_sin,date_cos,time,filtered_spike_durations,filtered_spike_magnitudes,filtered_time
0,3,"[2, 2, 3]","[3, 3, 4, 0]","[0.663, 0.493, 0.886, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.44636145833333335, 0.3711697916666667, 0.66...","[24, 34, 38]",-0.977848,0.209315,"[24, 34, 38, 0]","[1, 1, 2, 0]","[0.663, 0.493, 0.886, 0.0]","[24, 34, 38, 0]"
1,2,"[3, 2, 3]","[4, 5, 0, 0]","[0.933, 1.306, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.37326145833333335, 0.1712510416666667, 0.80...","[20, 36]",-0.974100,0.226116,"[20, 36, 0, 0]","[2, 3, 0, 0]","[0.933, 1.306, 0.0, 0.0]","[20, 36, 0, 0]"
2,2,"[2, 2, 3]","[6, 9, 0, 0]","[1.191, 1.085, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.35165416666666666, 0.17661875000000002, 0.7...","[20, 37]",-0.970064,0.242850,"[20, 37, 0, 0]","[4, 7, 0, 0]","[1.191, 1.085, 0.0, 0.0]","[20, 37, 0, 0]"
3,4,"[3, 2, 2, 3]","[3, 3, 3, 3]","[1.075, 0.551, 0.695, 1.164]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.392646875, 0.22844895833333334, 0.79748125]","[17, 23, 34, 38]",-0.965740,0.259512,"[17, 23, 34, 38]","[1, 1, 1, 1]","[1.075, 0.551, 0.695, 1.164]","[17, 23, 34, 38]"
4,3,"[2, 3, 2, 3]","[3, 3, 5, 0]","[0.408, 0.991, 1.413, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.45284687500000004, 0.3914020833333333, 0.65...","[18, 25, 36]",-0.961130,0.276097,"[18, 25, 36, 0]","[1, 1, 3, 0]","[0.408, 0.991, 1.413, 0.0]","[18, 25, 36, 0]"
5,2,"[2, 2]","[8, 3, 0, 0]","[0.784, 0.377, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.49727499999999997, 0.374875, 0.759745833333...","[19, 43]",-0.956235,0.292600,"[19, 43, 0, 0]","[6, 1, 0, 0]","[0.784, 0.377, 0.0, 0.0]","[19, 43, 0, 0]"
6,1,"[3, 2]","[7, 0, 0, 0]","[1.6720000000000002, 0.0, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.51195625, 0.274325, 0.8019531249999999]",[38],-0.951057,0.309017,"[38, 0, 0, 0]","[5, 0, 0, 0]","[1.6720000000000002, 0.0, 0.0, 0.0]","[38, 0, 0, 0]"
7,2,"[2, 2]","[3, 3, 0, 0]","[0.406, 0.822, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.47235, 0.09946458333333334, 0.9436062500000...","[34, 40]",-0.945596,0.325342,"[34, 40, 0, 0]","[1, 1, 0, 0]","[0.406, 0.822, 0.0, 0.0]","[34, 40, 0, 0]"
8,1,"[2, 3, 2]","[6, 16, 0, 0]","[0.798, 1.879, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.45446458333333334, 0.09321770833333333, 0.8...","[22, 33]",-0.939856,0.341571,"[22, 33, 0, 0]","[4, 0, 0, 0]","[0.798, 0.0, 0.0, 0.0]","[22, 0, 0, 0]"
9,2,"[2, 2, 2, 3, 2]","[7, 3, 17, 0]","[1.189, 0.657, 2.6060000000000003, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.4468854166666667, 0.2341895833333333, 0.917...","[20, 28, 32]",-0.933837,0.357698,"[20, 28, 32, 0]","[5, 1, 0, 0]","[1.189, 0.657, 0.0, 0.0]","[20, 28, 0, 0]"


In [11]:
# See the max value for spike_magnitude
max_magnitude = max(spike_df['spike_magnitudes'].apply(max))
print(max_magnitude)

61.311


In [12]:
all_values = [value for tensor in spike_df['filtered_time'] for value in tensor]

# Convert to a set to get the unique values and count them
unique_values = set(all_values)

# Number of unique values
num_unique_values = len(unique_values)
print(unique_values)

# Display the result
print(f"The number of different individual values in 'filtered_time' column is: {num_unique_values}")

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46}
The number of different individual values in 'filtered_time' column is: 47


In [13]:
all_values = [value for tensor in spike_df['filtered_spike_durations'] for value in tensor]

# Convert to a set to get the unique values and count them
unique_values = set(all_values)

# Number of unique values
num_unique_values = len(unique_values)

# Display the result
print(f"The number of different individual values in 'spike_durations' column is: {num_unique_values}")

The number of different individual values in 'spike_durations' column is: 8


In [14]:
class SpikeDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        spike_num = torch.tensor(row['spike_num'], dtype=torch.long).unsqueeze(-1)

        spike_durations = torch.tensor(row['filtered_spike_durations'], dtype=torch.long)
        
        spike_magnitudes = torch.tensor(row['filtered_spike_magnitudes'], dtype=torch.float32) / 50
        
        # Ensure that ID-statistics and other fields are properly parsed from strings to lists if necessary
        if isinstance(row['ID-statistics'], str):
            ID_statistics = torch.tensor(ast.literal_eval(row['ID-statistics']), dtype=torch.float32)
        else:
            ID_statistics = torch.tensor(row['ID-statistics'], dtype=torch.float32)

        if isinstance(row['weather'], str):
            weather = torch.tensor(ast.literal_eval(row['weather']), dtype=torch.float32)
        else:
            weather = torch.tensor(row['weather'], dtype=torch.float32)

        date_sin = torch.tensor(row['date_sin'], dtype=torch.float32).unsqueeze(-1)
        date_cos = torch.tensor(row['date_cos'], dtype=torch.float32).unsqueeze(-1)
        
        time = torch.tensor(row['filtered_time'], dtype=torch.long)

        return spike_num, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time

dataset = SpikeDataset(spike_df)

dataset_size = len(dataset)
train_size = int(0.1 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size  # Ensure the sum equals dataset_size

train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=True)
        

In [15]:
for idx, (spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time) in enumerate(train_dl):
    print('spike_nums shape:', spike_nums.shape)
    print('spike_durations shape:', spike_durations.shape)
    print('spike_magnitudes shape:', spike_magnitudes.shape)
    print('ID_statistics shape:', ID_statistics.shape)
    print('weather shape:', weather.shape)
    print('date_sin shape:', date_sin.shape)
    print('date_cos shape:', date_cos.shape)
    print('time shape:', time.shape)
    break

spike_nums shape: torch.Size([256, 1])
spike_durations shape: torch.Size([256, 4])
spike_magnitudes shape: torch.Size([256, 4])
ID_statistics shape: torch.Size([256, 6])
weather shape: torch.Size([256, 3])
date_sin shape: torch.Size([256, 1])
date_cos shape: torch.Size([256, 1])
time shape: torch.Size([256, 4])


# 2. Network

In [16]:
# class s_encoder(nn.Module):
#     def __init__(self, num_embed_size, hidden_size, duration_size, mag_size, time_size, output_size):
#         super(s_encoder, self).__init__()

#         self.hidden_size = hidden_size

#         self.num_embeddings = nn.Embedding(5, num_embed_size)
#         self.duration_embeddings = nn.Embedding(10, duration_size)
#         self.time_embeddings = nn.Embedding(47, time_size)

#         self.mag_net = nn.Sequential(
#             nn.Linear(4, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, mag_size),
#         )

#         self.fc = nn.Sequential(
#             nn.Linear(num_embed_size + duration_size + mag_size + time_size, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, output_size),
#         )
    
#     def forward(self, spike_nums, spike_durations, spike_magnitudes, time):

#         num = self.num_embeddings(spike_nums.squeeze())
        
#         duration = self.duration_embeddings(spike_durations)
#         duration = duration.mean(dim=1)

#         t = self.time_embeddings(time)
#         t = t.mean(dim=1)
    
#         magnitude = self.mag_net(spike_magnitudes)

#         x = torch.cat([num, duration, magnitude, t], dim=-1)
        
#         out = self.fc(x)
#         return out



# class s_decoder(nn.Module):
#     def __init__(self, latent_size, hidden_size, mix_size, id_size, weather_size, date_size):
#         super(s_decoder, self).__init__()
#         # Networks for processing ID_statistics, weather, and date_sin_cos
#         self.id_net = nn.Sequential(
#             nn.Linear(6, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, id_size),
#             nn.ReLU()
#         )

#         self.weather_net = nn.Sequential(
#             nn.Linear(3, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, weather_size),
#             nn.ReLU()
#         )

#         self.date_net = nn.Sequential(
#             nn.Linear(2, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, date_size),
#             nn.ReLU()
#         )


#         self.mix_net = nn.Sequential(
#             nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, mix_size),
#             nn.ReLU()
#         )

#         # Fully connected layers for reconstructing the output
#         self.spike_num_recon1 = nn.Sequential(
#             nn.Linear(latent_size + 11, hidden_size),
#             nn.ReLU(),
#             nn.Dropout(0.5), 
#             nn.Linear(hidden_size, hidden_size),
#             nn.ReLU(), 
#             nn.Dropout(0.5), 
#             nn.Linear(hidden_size, hidden_size),
#             nn.ReLU(), 
#         )

#         self.spike_num_recon2 = nn.Sequential(
#             nn.Linear(hidden_size + latent_size + 11 + mix_size, hidden_size), #  + 32
#             nn.ReLU(),
#             nn.Dropout(0.5), 
#             nn.Linear(hidden_size, hidden_size),
#             nn.ReLU(),
#             nn.Dropout(0.5),
#             nn.Linear(hidden_size, 5),
#         )

#         self.spike_durations_recon1 = nn.Sequential(
#             nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, 4),
#         )

#         self.spike_durations_recon2 = nn.Sequential(
#             nn.Linear(7, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, 7)
#         )

#         self.spike_magnitudes_recon1 = nn.Sequential(
#             nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, 16),
#             nn.ReLU()
#         )

#         self.spike_magnitudes_recon2 = nn.Sequential(
#             nn.Linear(16 + latent_size + 11, hidden_size),
#             nn.Softplus(),
#             nn.Linear(hidden_size, 4),
#             nn.Softplus()
#         )

#         self.time_recon1 = nn.Sequential(
#             nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, 4)
#         )
        
#         self.time_recon2 = nn.Sequential(
#             nn.Linear(47, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, hidden_size),
#             nn.ReLU(),
#             nn.Linear(hidden_size, 47),
#         )

    
#     def forward(self, z, ID_statistics, weather, date_sin, date_cos):
        
#         id, w, date_sin_cos = self.id_net(ID_statistics), self.weather_net(weather), self.date_net(torch.cat([date_sin, date_cos], dim=-1))

#         mix = self.mix_net(torch.cat([z, id, w, date_sin_cos], dim=-1))
#         concat = torch.cat([z, id, w, date_sin_cos], dim=-1)

#         concat_original = torch.cat([z, ID_statistics, weather, date_sin, date_cos], dim=-1)

#         # Reconstruct the output
#         # spike_num_recon1 = self.spike_num_recon1(concat_original)
#         spike_num_recon1 = self.spike_num_recon1(concat_original)
#         spike_num_recon = self.spike_num_recon2(torch.cat([spike_num_recon1, concat_original, mix], dim=-1))
        
#         spike_durations_recon1 = self.spike_durations_recon1(concat)
#         spike_durations_recon2 = spike_durations_recon1.unsqueeze(-1).expand(-1, -1, 7)  
#         spike_durations_recon = self.spike_durations_recon2(spike_durations_recon2)

#         spike_magnitudes_recon = self.spike_magnitudes_recon1(concat)
#         spike_magnitudes_recon = self.spike_magnitudes_recon2(torch.cat([spike_magnitudes_recon, concat_original], dim=-1))
        
#         time_recon1 = self.time_recon1(concat)
#         time_recon2 = time_recon1.unsqueeze(-1).expand(-1, -1, 47)
#         time_recon = self.time_recon2(time_recon2)

#         return spike_num_recon, spike_durations_recon, spike_magnitudes_recon, time_recon


# class s_VAE(nn.Module):
#     def __init__(self, num_embed_size, hidden_size_e, duration_size, mag_size, time_size, latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size):
#         super(s_VAE, self).__init__()

#         self.encoder = s_encoder(num_embed_size, hidden_size_e, duration_size, mag_size, time_size, output_size=latent_size * 2)
#         self.decoder = s_decoder(latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size)

#     def reparameterize(self, mu, logvar):
#         std = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(std)
#         return mu + eps * std
        
#     def forward(self, spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time_sin_cos):
#         encoder_out = self.encoder(spike_nums, spike_durations, spike_magnitudes, time_sin_cos)
#         mu, logvar = encoder_out.chunk(2, dim=1)

#         # Reparameterization trick to sample from the latent space
#         z = self.reparameterize(mu, logvar)

#         # Decode the latent variable to reconstruct the input
#         spike_num_recon, spike_durations_recon, spike_magnitudes_recon, time_recon = self.decoder(z, ID_statistics, weather, date_sin, date_cos)

#         return spike_num_recon, spike_durations_recon, spike_magnitudes_recon, time_recon, mu, logvar

#     def init_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
#                 nn.init.constant_(m.bias, 1e-2)


In [17]:
class s_encoder(nn.Module):
    def __init__(self, num_embed_size, hidden_size, duration_size, mag_size, time_size, output_size):
        super(s_encoder, self).__init__()

        self.hidden_size = hidden_size

        self.num_embeddings = nn.Embedding(5, num_embed_size)
        self.duration_embeddings = nn.Embedding(10, duration_size)
        self.time_embeddings = nn.Embedding(47, time_size)

        self.num_net = nn.Sequential(
            nn.Linear(num_embed_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
        )

        self.duration_net = nn.Sequential(
            nn.Linear(duration_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
        )

        self.mag_net = nn.Sequential(
            nn.Linear(4, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, mag_size),
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size + hidden_size + mag_size + time_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )
    
    def forward(self, spike_nums, spike_durations, spike_magnitudes, time):

        num = self.num_embeddings(spike_nums.squeeze())
        num = self.num_net(num)
        
        duration = self.duration_embeddings(spike_durations)
        duration = duration.mean(dim=1)
        duration = self.duration_net(duration)
        

        t = self.time_embeddings(time)
        t = t.mean(dim=1)
    
        magnitude = self.mag_net(spike_magnitudes)

        x = torch.cat([num, duration, magnitude, t], dim=-1)
        
        out = self.fc(x)
        return out

class s_decoder_num(nn.Module):
    def __init__(self, latent_size, hidden_size, mix_size, id_size, weather_size, date_size):
        super(s_decoder_num, self).__init__()
        # Networks for processing ID_statistics, weather, and date_sin_cos
        self.id_net = nn.Sequential(
            nn.Linear(6, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, id_size),
        )

        self.weather_net = nn.Sequential(
            nn.Linear(3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, weather_size),
        )

        self.date_net = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, date_size),
        )

        self.mix_net = nn.Sequential(
            nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, mix_size),
        )

        self.spike_num_recon1 = nn.Sequential(
            nn.Linear(latent_size + 11, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(), 
            nn.Dropout(0.5), 
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(), 
            nn.Dropout(0.5), 
            nn.Linear(hidden_size, hidden_size),
        )

        self.spike_num_recon2 = nn.Sequential(
            nn.Linear(hidden_size + latent_size + 11 + mix_size, hidden_size), #  + 32
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_size, 5),
        )

        self.spike_num_recon3 = nn.Sequential(
            nn.Linear(5 + latent_size + id_size + weather_size + date_size , hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 5),
            nn.ReLU(),
            nn.Linear(5, 5),
            nn.Linear(5, 5)
        )
    
    def forward(self, z, ID_statistics, weather, date_sin, date_cos):
        
        id, w, date_sin_cos = self.id_net(ID_statistics), self.weather_net(weather), self.date_net(torch.cat([date_sin, date_cos], dim=-1))

        mix = self.mix_net(torch.cat([z, id, w, date_sin_cos], dim=-1))
        concat = torch.cat([z, id, w, date_sin_cos], dim=-1)
        concat_original = torch.cat([z, ID_statistics, weather, date_sin, date_cos], dim=-1)

        # Reconstruct the output
        # spike_num_recon1 = self.spike_num_recon1(concat_original)
        spike_num_recon1 = self.spike_num_recon1(concat_original)
        spike_num_recon2 = self.spike_num_recon2(torch.cat([spike_num_recon1, concat_original, mix], dim=-1))
        spike_num_recon = self.spike_num_recon3(torch.cat([spike_num_recon2, concat], dim=-1))

        return spike_num_recon

    
class s_decoder_duration(nn.Module):
    def __init__(self, latent_size, hidden_size, mix_size, id_size, weather_size, date_size):
        super(s_decoder_duration, self).__init__()
        # Networks for processing ID_statistics, weather, and date_sin_cos
        self.id_net = nn.Sequential(
            nn.Linear(6, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, id_size),
            nn.ReLU()
        )

        self.weather_net = nn.Sequential(
            nn.Linear(3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, weather_size),
            nn.ReLU()
        )

        self.date_net = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, date_size),
            nn.ReLU()
        )


        self.mix_net = nn.Sequential(
            nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, mix_size),
            nn.ReLU()
        )
        self.spike_durations_recon1 = nn.Sequential(
            nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 4),
        )

        self.spike_durations_recon2 = nn.Sequential(
            nn.Linear(7, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 7)
        )
        
    
    def forward(self, z, ID_statistics, weather, date_sin, date_cos):
        
        id, w, date_sin_cos = self.id_net(ID_statistics), self.weather_net(weather), self.date_net(torch.cat([date_sin, date_cos], dim=-1))

        mix = self.mix_net(torch.cat([z, id, w, date_sin_cos], dim=-1))
        concat = torch.cat([z, id, w, date_sin_cos], dim=-1)
        concat_original = torch.cat([z, ID_statistics, weather, date_sin, date_cos], dim=-1)

        # Reconstruct the output
        # spike_num_recon1 = self.spike_num_recon1(concat_original)
        spike_durations_recon1 = self.spike_durations_recon1(concat)
        spike_durations_recon2 = spike_durations_recon1.unsqueeze(-1).expand(-1, -1, 7)  
        spike_durations_recon = self.spike_durations_recon2(spike_durations_recon2)

        return spike_durations_recon

class s_decoder_magnitude(nn.Module):
    def __init__(self, latent_size, hidden_size, mix_size, id_size, weather_size, date_size):
        super(s_decoder_magnitude, self).__init__()
        # Networks for processing ID_statistics, weather, and date_sin_cos
        self.id_net = nn.Sequential(
            nn.Linear(6, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, id_size),
            nn.ReLU()
        )

        self.weather_net = nn.Sequential(
            nn.Linear(3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, weather_size),
            nn.ReLU()
        )

        self.date_net = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, date_size),
            nn.ReLU()
        )


        self.mix_net = nn.Sequential(
            nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, mix_size),
            nn.ReLU()
        )

        self.spike_magnitudes_recon1 = nn.Sequential(
            nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 16),
            nn.ReLU()
        )

        self.spike_magnitudes_recon2 = nn.Sequential(
            nn.Linear(16 + latent_size + 11, hidden_size),
            nn.Softplus(),
            nn.Linear(hidden_size, 4),
            nn.Softplus()
        )
    
    def forward(self, z, ID_statistics, weather, date_sin, date_cos):
        
        id, w, date_sin_cos = self.id_net(ID_statistics), self.weather_net(weather), self.date_net(torch.cat([date_sin, date_cos], dim=-1))

        mix = self.mix_net(torch.cat([z, id, w, date_sin_cos], dim=-1))
        concat = torch.cat([z, id, w, date_sin_cos], dim=-1)
        concat_original = torch.cat([z, ID_statistics, weather, date_sin, date_cos], dim=-1)

        spike_magnitudes_recon = self.spike_magnitudes_recon1(concat)
        spike_magnitudes_recon = self.spike_magnitudes_recon2(torch.cat([spike_magnitudes_recon, concat_original], dim=-1))
        
        return spike_magnitudes_recon
    
class s_decoder_time(nn.Module):
    def __init__(self, latent_size, hidden_size, mix_size, id_size, weather_size, date_size):
        super(s_decoder_time, self).__init__()
        # Networks for processing ID_statistics, weather, and date_sin_cos
        self.id_net = nn.Sequential(
            nn.Linear(6, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, id_size),
            nn.ReLU()
        )

        self.weather_net = nn.Sequential(
            nn.Linear(3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, weather_size),
            nn.ReLU()
        )

        self.date_net = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, date_size),
            nn.ReLU()
        )

        self.mix_net = nn.Sequential(
            nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, mix_size),
            nn.ReLU()
        )
        self.time_recon1 = nn.Sequential(
            nn.Linear(latent_size + id_size + weather_size + date_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 4)
        )
        
        self.time_recon2 = nn.Sequential(
            nn.Linear(47, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 47),
        )
    
    def forward(self, z, ID_statistics, weather, date_sin, date_cos):
        
        id, w, date_sin_cos = self.id_net(ID_statistics), self.weather_net(weather), self.date_net(torch.cat([date_sin, date_cos], dim=-1))

        mix = self.mix_net(torch.cat([z, id, w, date_sin_cos], dim=-1))
        concat = torch.cat([z, id, w, date_sin_cos], dim=-1)
        concat_original = torch.cat([z, ID_statistics, weather, date_sin, date_cos], dim=-1)

        time_recon1 = self.time_recon1(concat)
        time_recon2 = time_recon1.unsqueeze(-1).expand(-1, -1, 47)
        time_recon = self.time_recon2(time_recon2)

        return time_recon
        

class s_VAE(nn.Module):
    def __init__(self, num_embed_size, hidden_size_e, duration_size, mag_size, time_size, latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size):
        super(s_VAE, self).__init__()

        self.encoder_num = s_encoder(num_embed_size, hidden_size_e, duration_size, mag_size, time_size, output_size=latent_size * 2)
        self.encoder_duration = s_encoder(num_embed_size, hidden_size_e, duration_size, mag_size, time_size, output_size=latent_size * 2)
        self.encoder_magnitude = s_encoder(num_embed_size, hidden_size_e, duration_size, mag_size, time_size, output_size=latent_size * 2)
        self.encoder_time = s_encoder(num_embed_size, hidden_size_e, duration_size, mag_size, time_size, output_size=latent_size * 2)

        self.decoder_num = s_decoder_num(latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size)
        self.decoder_duration = s_decoder_duration(latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size)
        self.decoder_magnitude = s_decoder_magnitude(latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size)
        self.decoder_time = s_decoder_time(latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size)


    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def forward(self, spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time_sin_cos):
        encoder_out_num = self.encoder_num(spike_nums, spike_durations, spike_magnitudes, time_sin_cos)
        encoder_out_duration = self.encoder_duration(spike_nums, spike_durations, spike_magnitudes, time_sin_cos)
        encoder_out_magnitude = self.encoder_magnitude(spike_nums, spike_durations, spike_magnitudes, time_sin_cos)
        encoder_out_time = self.encoder_time(spike_nums, spike_durations, spike_magnitudes, time_sin_cos)

        mu_num, logvar_num = encoder_out_num.chunk(2, dim=1)
        mu_duration, logvar_duration = encoder_out_duration.chunk(2, dim=1)
        mu_magnitude, logvar_magnitude = encoder_out_magnitude.chunk(2, dim=1)
        mu_time, logvar_time = encoder_out_time.chunk(2, dim=1)

        z_num = self.reparameterize(mu_num, logvar_num)
        z_duration = self.reparameterize(mu_duration, logvar_duration)
        z_magnitude = self.reparameterize(mu_magnitude, logvar_magnitude)
        z_time = self.reparameterize(mu_time, logvar_time)

        # Decode the latent variable to reconstruct the input
        spike_num_recon = self.decoder_num(z_num, ID_statistics, weather, date_sin, date_cos)
        spike_durations_recon = self.decoder_duration(z_duration, ID_statistics, weather, date_sin, date_cos)
        spike_magnitudes_recon = self.decoder_magnitude(z_magnitude, ID_statistics, weather, date_sin, date_cos)
        time_recon = self.decoder_time(z_time, ID_statistics, weather, date_sin, date_cos)

        return spike_num_recon, spike_durations_recon, spike_magnitudes_recon, time_recon, mu_num, logvar_num, mu_duration, logvar_duration, mu_magnitude, logvar_magnitude, mu_time, logvar_time

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 1e-2)


# 4. Training

In [18]:
# Define hyperparameters
from sklearn.utils.class_weight import compute_class_weight
num_embed_size = 64
hidden_size_e = 128
duration_size = 32
mag_size = 32
time_size = 32
 
latent_size = 64

hidden_size_d = 256
mix_size = 256
id_size = 256
weather_size = 64
date_size = 32

# Initialize the model
model = s_VAE(num_embed_size, hidden_size_e, duration_size, mag_size, time_size, latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size).to(device)
model.init_weights()

# Define the loss function and optimizer
criterion_mse = nn.MSELoss()
criterion_ce = nn.CrossEntropyLoss()

optimizer_num = optim.AdamW(model.parameters(), lr=1e-6)
optimizer_durations = optim.Adam(model.parameters(), lr=1e-4)
optimizer_magnitudes = optim.Adam(model.parameters(), lr=1e-4)
optimizer_time = optim.Adam(model.parameters(), lr=1e-4)
optimizer_kld = optim.Adam(model.parameters(), lr=1e-4)


In [21]:
# Training loop
num_epochs = 500
patience = 10
best_loss = float('inf')
patience_counter = 0
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    num_loss = 0.0
    dur_loss = 0.0
    mag_loss = 0.0
    t_loss = 0.0

    for idx, (spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time) in enumerate(train_dl):

        spike_nums = spike_nums.to(device)
        spike_durations = spike_durations.to(device)
        spike_magnitudes = spike_magnitudes.to(device)
        ID_statistics = ID_statistics.to(device)
        weather = weather.to(device)
        date_sin = date_sin.to(device)
        date_cos = date_cos.to(device)
        time = time.to(device)
        
        optimizer_num.zero_grad()
        optimizer_durations.zero_grad()
        optimizer_magnitudes.zero_grad()
        optimizer_time.zero_grad()
        
        # spike_num_recon, spike_durations_recon, spike_magnitudes_recon, time_recon, mu, logvar = model(
        #     spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time
        # )
        spike_num_recon, spike_durations_recon, spike_magnitudes_recon, time_recon, mu_num, logvar_num, mu_duration, logvar_duration, mu_magnitude, logvar_magnitude, mu_time, logvar_time = model(
            spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time
        )

        spike_num_label = spike_nums.view(-1)
        spike_nums_cpu = spike_nums.cpu().numpy().ravel()
        class_weights = compute_class_weight('balanced', classes=[0, 1, 2, 3, 4], y=spike_nums_cpu)
        class_weights = torch.tensor(class_weights, dtype=torch.float, device=spike_nums.device)
        criterion_ce_x = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        spike_num_loss = criterion_ce_x(spike_num_recon, spike_num_label)
        kld_num = -0.5 * torch.sum(1 + logvar_num - mu_num.pow(2) - logvar_num.exp())
        spike_num_loss += kld_num
        spike_num_loss.backward()
        optimizer_num.step()
        num_loss += spike_num_loss.item()

        mask = (spike_durations != 0)
        sd_m = spike_durations[mask].long()
        mask = mask.unsqueeze(-1).expand(-1, -1, 7)
        sdr_m = spike_durations_recon[mask].view(-1, 7)
        # sd_m = spike_durations
        # sdr_m = spike_durations_recon
        sd_m_cpu = sd_m.cpu().numpy().ravel()
        unique_classes = np.unique(sd_m_cpu)
        class_weights = compute_class_weight('balanced', classes=np.unique(sd_m_cpu), y=sd_m_cpu)
        full_class_weights = np.ones(7, dtype=np.float32)  # Change to the number of classes you have
        # Assign the computed weights to the appropriate positions in the full weight array
        for i, cls in enumerate(unique_classes):
            full_class_weights[cls-1] = class_weights[i]
            
        class_weights_tensor = torch.tensor(full_class_weights, dtype=torch.float, device=sd_m.device)
        criterion_ce_x = nn.CrossEntropyLoss(weight=class_weights_tensor)
        spike_duration_loss = criterion_ce_x(sdr_m, sd_m-1)
        kld_duration = -0.5 * torch.sum(1 + logvar_duration - mu_duration.pow(2) - logvar_duration.exp())
        spike_duration_loss += kld_duration
        spike_duration_loss.backward()
        optimizer_durations.step()
        dur_loss += spike_duration_loss.item()

        mask = (spike_magnitudes != 0)
        sm_m, smr_m = spike_magnitudes[mask], spike_magnitudes_recon[mask]
        spike_magnitude_loss = criterion_mse(smr_m, sm_m)
        kld_mag = -0.5 * torch.sum(1 + logvar_magnitude - mu_magnitude.pow(2) - logvar_magnitude.exp())
        spike_magnitude_loss += kld_mag
        spike_magnitude_loss.backward()
        optimizer_magnitudes.step()
        mag_loss += spike_magnitude_loss.item()

        mask = (time != 0)
        t_m = time[mask].long()
        mask = mask.unsqueeze(-1).expand(-1, -1, 47)
        tr_m = time_recon[mask].view(-1, 47)
        time_loss = criterion_ce(tr_m, t_m)
        kld_time = -0.5 * torch.sum(1 + logvar_time - mu_time.pow(2) - logvar_time.exp())
        time_loss += kld_time
        time_loss.backward()
        optimizer_time.step()
        t_loss += time_loss.item()

        train_loss += spike_num_loss.item() + spike_duration_loss.item() + spike_magnitude_loss.item() + time_loss.item()

    # scheduler.step(train_loss / len(train_dl))

    if train_loss < best_loss:
        best_loss = train_loss
        best_model = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping: patience limit reached at epoch {epoch+1}")
        break
    
    print(f"Epoch {epoch+1}, Loss: {train_loss / len(train_dl)}")
    print(f"Num Loss: {num_loss / len(train_dl)}")
    print(f"Dur Loss: {dur_loss / len(train_dl)}")
    print(f"Mag Loss: {mag_loss / len(train_dl)}")
    print(f"Time Loss: {time_loss / len(train_dl)}")


Epoch 1, Loss: 6.990753282938079
Num Loss: 1.667657046150743
Dur Loss: 1.9412777969711705
Mag Loss: 0.000533648480440637
Time Loss: 0.002989509841427207
Epoch 2, Loss: 6.991025225832428
Num Loss: 1.667570812451212
Dur Loss: 1.9415669514421832
Mag Loss: 0.0005336634826738174
Time Loss: 0.002974089002236724
Epoch 3, Loss: 6.991058082907081
Num Loss: 1.6674754573587787
Dur Loss: 1.9416476315573643
Mag Loss: 0.0005337897646281737
Time Loss: 0.003018416231498122
Epoch 4, Loss: 6.991001961372814
Num Loss: 1.6679286151601558
Dur Loss: 1.941430259796611
Mag Loss: 0.0005337109224791149
Time Loss: 0.002932655392214656
Epoch 5, Loss: 6.991280001154212
Num Loss: 1.6678524778600325
Dur Loss: 1.941690291019908
Mag Loss: 0.0005338870256979901
Time Loss: 0.002957309363409877


KeyboardInterrupt: 

In [22]:
# Save the model
torch.save(model.state_dict(), 's_vae.pth')
print("Model saved successfully!")

Model saved successfully!


# 5. Testing

In [23]:
# # Load the model
# # Define hyperparameters
# num_embed_size = 8
# hidden_size_e = 128
# duration_size = 32
# mag_size = 32
# time_size = 32
 
# latent_size = 1024

# z_size = 32
# hidden_size_d = 256
# mix_size = 256
# id_size = 256
# weather_size = 64
# date_size = 32

# Initialize the model
model = s_VAE(num_embed_size, hidden_size_e, duration_size, mag_size, time_size, latent_size, hidden_size_d, mix_size, id_size, weather_size, date_size).to(device)
model.load_state_dict(torch.load('s_vae.pth'))
model.eval()


s_VAE(
  (encoder_num): s_encoder(
    (num_embeddings): Embedding(5, 64)
    (duration_embeddings): Embedding(10, 32)
    (time_embeddings): Embedding(47, 32)
    (num_net): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
    (duration_net): Sequential(
      (0): Linear(in_features=32, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
    (mag_net): Sequential(
      (0): Linear(in_features=4, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=32, bias=True)
    )
    (fc): Sequential(
      (0): Linear(in_features=320, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): ReLU()
      (4): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (encoder_duration): s_encoder(
    (num_e

In [33]:
random_idx = np.random.randint(0, len(test_ds))
spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time = test_ds[random_idx]

spike_nums = spike_nums.to(device)
spike_durations = spike_durations.to(device)
spike_magnitudes = spike_magnitudes.to(device)
ID_statistics = ID_statistics.to(device)
weather = weather.to(device)
date_sin = date_sin.to(device)
date_cos = date_cos.to(device)
time = time.to(device)

spike_durations = [duration for duration in spike_durations.squeeze().tolist() if duration != 0.0]
spike_durations = [int(x +2 ) for x in spike_durations]
spike_magnitudes = [magnitude for magnitude in spike_magnitudes.squeeze().tolist() if magnitude != 0.0]
spike_magnitudes = [round(x * 50, 3)  for x in spike_magnitudes]
time = [time for time in time.squeeze().tolist() if time != 0]
print('Real')
print(f"num: {spike_nums.item()}") #  | durations: {spike_durations} | magnitudes:{spike_magnitudes} | times: {time}

# (1, latent_size) noise
noise = torch.randn(latent_size).to(device)

spike_num_recon = model.decoder_num(noise, ID_statistics, weather, date_sin, date_cos)
# spike_durations_recon = model.decoder_duration(noise, ID_statistics, weather, date_sin, date_cos)
# spike_magnitudes_recon = model.decoder_magnitude(noise, ID_statistics, weather, date_sin, date_cos)
# time_recon = model.decoder_time(noise, ID_statistics, weather, date_sin, date_cos)

syn_spike_num = spike_num_recon.argmax(dim=-1).item()
# print(syn_spike_num)

syn_spike_durations = spike_durations_recon[:, :syn_spike_num]
# print(syn_spike_durations.shape)
syn_spike_durations = syn_spike_durations.squeeze()
syn_spike_durations = torch.argmax(syn_spike_durations, axis=-1) + 3
syn_spike_durations = syn_spike_durations.tolist()

syn_spike_magnitudes = spike_magnitudes_recon.squeeze().tolist()[:syn_spike_num]
syn_spike_magnitudes = [x * 50 for x in syn_spike_magnitudes]

syn_time = time_recon[:, :syn_spike_num].squeeze()
syn_time = torch.argmax(syn_time, axis=-1).tolist()
# if type(syn_time) == list:
#     syn_time = [x * 48 for x in syn_time]
#     syn_time = [math.ceil(x) for x in syn_time]
# else:
#     syn_time = syn_time*48
#     syn_time = math.ceil(syn_time)

print('Synthetic')
print(f"num: {syn_spike_num} ") # | durations: {syn_spike_durations} | magnitudes: {syn_spike_magnitudes} | times: {syn_time}

Real
num: 2
Synthetic
num: 4 


: 